# 4. Feature Engineering

This notebook processes the cleaned training and validation data (at the subsegment level) to create target lags, rolling window statistics, calendar features, and macroeconomic indicators.



In [1]:
import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

## 1. Load Data
We combine training and validation sets temporarily so that lags and rolling statistics compute correctly across the split boundaries.



In [ ]:
data_dir = '../data/prepared'
train_path = os.path.join(data_dir, 'training_subsegment.parquet')
val_path = os.path.join(data_dir, 'validation_subsegment.parquet')
macro_path = os.path.join(data_dir, 'macro_data_clean.parquet')

if not os.path.exists(train_path):
    print("Prepared data not found. Please run 2)Data_Preparation.ipynb first.")
else:
    df_train = pd.read_parquet(train_path)
    df_val = pd.read_parquet(val_path)
    df_macro = pd.read_parquet(macro_path)
    
    # Combine train and val to prevent data loss at the split boundary for lag features
    df_train['is_train'] = True
    df_val['is_train'] = False
    
    df_full = pd.concat([df_train, df_val], ignore_index=True)
    df_full.sort_values(by=['TGL Business Subsegment', 'Anon Period'], inplace=True)
    df_full.reset_index(drop=True, inplace=True)
    print("Data loaded successfully. Combined shape:", df_full.shape)

Data loaded successfully. Combined shape: (4952, 84)


## 2. Calendar Features
Extracting relative month, quarter, and using Anon Period directly as a time index proxy to help the model learn general trends.



In [ ]:
# Assume Anon Period roughly starts mapping sequentially month by month
# We extract relative months (1-12) to model seasonality
df_full['Month'] = ((df_full['Anon Period'] - 1) % 12) + 1
df_full['Quarter'] = ((df_full['Month'] - 1) // 3) + 1
df_full['Period_Index'] = df_full['Anon Period']

print("Calendar features created.")
display(df_full[['Anon Period', 'Month', 'Quarter', 'Period_Index']].head())

Calendar features created.


,Anon Period,Month,Quarter,Period_Index
0,1,1,1,1
1,2,2,1,2
2,3,3,1,3
3,4,4,2,4
4,5,5,2,5


## 3. Target Lag Features & Rolling Window Statistics (Per Subsegment)
Here we generate lag features and moving averages using past periods, which will give LightGBM autoregressive capability.



In [ ]:
targets = ['Orders cons. (anon)', 'Revenue cons. (anon)']
lags = [1, 2, 3, 6, 12]
rolling_windows = [3, 6, 12]

# Ensure sorting is consistent before applying group shifts
df_full.sort_values(by=['TGL Business Subsegment', 'Anon Period'], inplace=True)

# Generate Lag Features
for target in targets:
    for lag in lags:
        df_full[f'{target}_Lag_{lag}'] = df_full.groupby('TGL Business Subsegment')[target].shift(lag)

# Generate Rolling Statistics
for target in targets:
    for window in rolling_windows:
        # Crucial: Use Lag_1 as the base for rolling stats to prevent data leakage of the current period!
        df_full[f'{target}_Rolling_Mean_{window}'] = df_full.groupby('TGL Business Subsegment')[f'{target}_Lag_1'].transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )
        if window >= 3:
            df_full[f'{target}_Rolling_Std_{window}'] = df_full.groupby('TGL Business Subsegment')[f'{target}_Lag_1'].transform(
                lambda x: x.rolling(window, min_periods=2).std()
            )

print("Lag and rolling features created.")
display(df_full[[c for c in df_full.columns if 'Lag' in c or 'Rolling' in c]].head())

Lag and rolling features created.


,Orders cons. (anon)_Lag_1,Orders cons. (anon)_Lag_2,Orders cons. (anon)_Lag_3,Orders cons. (anon)_Lag_6,Orders cons. (anon)_Lag_12,Revenue cons. (anon)_Lag_1,Revenue cons. (anon)_Lag_2,Revenue cons. (anon)_Lag_3,Revenue cons. (anon)_Lag_6,Revenue cons. (anon)_Lag_12,Orders cons. (anon)_Rolling_Mean_3,Orders cons. (anon)_Rolling_Std_3,Orders cons. (anon)_Rolling_Mean_6,Orders cons. (anon)_Rolling_Std_6,Orders cons. (anon)_Rolling_Mean_12,Orders cons. (anon)_Rolling_Std_12,Revenue cons. (anon)_Rolling_Mean_3,Revenue cons. (anon)_Rolling_Std_3,Revenue cons. (anon)_Rolling_Mean_6,Revenue cons. (anon)_Rolling_Std_6,Revenue cons. (anon)_Rolling_Mean_12,Revenue cons. (anon)_Rolling_Std_12
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.0,NaN,NaN,NaN,NaN,-390277.0,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,0.0,NaN,-390277.000000,NaN,-390277.000000,NaN,-390277.000000,NaN
2,0.0,0.0,NaN,NaN,NaN,334.0,-390277.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,-194971.500000,2.762037e+05,-194971.500000,276203.686906,-194971.500000,276203.686906
3,0.0,0.0,0.0,NaN,NaN,334.0,334.0,-390277.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,-129869.666667,2.255194e+05,-129869.666667,225519.365998,-129869.666667,225519.365998
4,0.0,0.0,0.0,NaN,NaN,1811758.0,334.0,334.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,604142.000000,1.045826e+06,355537.250000,988122.203282,355537.250000,988122.203282


## 4. Macroeconomic Indicators
We extract a subset of the most critical macro variables (GDP and Inflation) and compute their lags and Month-over-Month (MoM) differences to capture economic trajectory without creating an explosive amount of features.



In [ ]:
# Find key macro columns in the pure macro dataset
macro_cols = [c for c in df_macro.columns if c != 'Period']
key_macro_cols = [c for c in macro_cols if 'GDP' in c or 'Inflation' in c]

# Compute on unique periods dataset to avoid repetition, then join
df_macro_unique = df_macro.copy().sort_values('Period')

for col in key_macro_cols:
    df_macro_unique[f'{col}_Lag_1'] = df_macro_unique[col].shift(1)
    df_macro_unique[f'{col}_Lag_3'] = df_macro_unique[col].shift(3)
    df_macro_unique[f'{col}_MoM_Diff'] = df_macro_unique[col] - df_macro_unique[f'{col}_Lag_1']

# Filter to keep only the newly created lag and diff macro features
new_macro_features = [c for c in df_macro_unique.columns if '_Lag_' in c or '_MoM_Diff' in c]

# Merge these new features back into df_full
df_full = df_full.merge(df_macro_unique[['Period'] + new_macro_features], left_on='Anon Period', right_on='Period', how='left')
df_full.drop(columns=['Period'], inplace=True, errors='ignore')

print(f"Macro features added: {len(new_macro_features)} new columns.")

Macro features added: 114 new columns.


## 5. Categorical Types
Set categorical columns correctly so LightGBM can optimally process them natively as categoricals.



In [ ]:
cat_cols = ['TGL Business Unit', 'TGL Business Segment', 'TGL Business Subsegment']

for col in cat_cols:
    df_full[col] = df_full[col].astype('category')


print("Categorical types casted.")

Categorical types casted.


## 6. Train/Validation Split & Saving
Split the dataset back and save to the features directory.



In [ ]:
# Split back to Training and Validation
df_train_fe = df_full[df_full['is_train'] == True].drop(columns=['is_train'])
df_val_fe = df_full[df_full['is_train'] == False].drop(columns=['is_train'])

# Ensure no NaNs exist in target variables and base columns (NaNs in Lags are expected for the first few periods)
output_dir = '../data/features'
os.makedirs(output_dir, exist_ok=True)

train_out = os.path.join(output_dir, 'training_subsegment_features.parquet')
val_out = os.path.join(output_dir, 'validation_subsegment_features.parquet')

df_train_fe.to_parquet(train_out, index=False)
df_val_fe.to_parquet(val_out, index=False)

print(f"Feature engineering complete. Data saved to {output_dir}")
print("Train shape:", df_train_fe.shape)
print("Val shape:", df_val_fe.shape)

Feature engineering complete. Data saved to ../data/features
Train shape: (4237, 222)
Val shape: (715, 222)


In [ ]:
df_train_fe.head()

,Anon Period,TGL Business Unit,TGL Business Segment,TGL Business Subsegment,Orders cons. (anon),Revenue cons. (anon),China_Core_Inflation_Rate,China_Exports,China_GDP,China_GDP_from_Construction,China_GDP_from_Manufacturing,China_Industrial_Production,China_Industrial_Production_Mom,China_Inflation_Rate,China_Interest_Rate,China_Steel_Production,France_Core_Inflation_Rate,France_Exports,France_GDP,France_GDP_from_Construction,France_GDP_from_Manufacturing,France_Industrial_Production,France_Industrial_Production_Mom,France_Inflation_Rate,France_Interest_Rate,France_Steel_Production,Germany_Core_Inflation_Rate,Germany_Exports,Germany_GDP,Germany_GDP_from_Construction,Germany_GDP_from_Manufacturing,Germany_Industrial_Production,Germany_Industrial_Production_Mom,Germany_Inflation_Rate,Germany_Interest_Rate,Germany_Steel_Production,Italy_Core_Inflation_Rate,Italy_Exports,Italy_GDP,Italy_GDP_from_Construction,Italy_GDP_from_Manufacturing,Italy_Industrial_Production,Italy_Industrial_Production_Mom,Italy_Inflation_Rate,Italy_Interest_Rate,Italy_Steel_Production,Japan_Core_Inflation_Rate,Japan_Exports,Japan_GDP,Japan_GDP_from_Construction,Japan_GDP_from_Manufacturing,Japan_Industrial_Production,Japan_Industrial_Production_Mom,Japan_Inflation_Rate,Japan_Interest_Rate,Japan_Steel_Production,Switzerland_Core_Inflation_Rate,Switzerland_Exports,Switzerland_GDP,Switzerland_Industrial_Production,Switzerland_Industrial_Production_Mom,Switzerland_Inflation_Rate,Switzerland_Interest_Rate,United_Kingdom_Core_Inflation_Rate,United_Kingdom_Exports,United_Kingdom_GDP,United_Kingdom_GDP_from_Construction,United_Kingdom_GDP_from_Manufacturing,United_Kingdom_Industrial_Production,United_Kingdom_Industrial_Production_Mom,United_Kingdom_Inflation_Rate,United_Kingdom_Interest_Rate,United_Kingdom_Steel_Production,United_States_Core_Inflation_Rate,United_States_Exports,United_States_GDP,United_States_GDP_from_Construction,United_States_GDP_from_Manufacturing,United_States_Industrial_Production,United_States_Industrial_Production_Mom,United_States_Inflation_Rate,United_States_Interest_Rate,United_States_Steel_Production,Month,Quarter,Period_Index,Orders cons. (anon)_Lag_1,Orders cons. (anon)_Lag_2,Orders cons. (anon)_Lag_3,Orders cons. (anon)_Lag_6,Orders cons. (anon)_Lag_12,Revenue cons. (anon)_Lag_1,Revenue cons. (anon)_Lag_2,Revenue cons. (anon)_Lag_3,Revenue cons. (anon)_Lag_6,Revenue cons. (anon)_Lag_12,Orders cons. (anon)_Rolling_Mean_3,Orders cons. (anon)_Rolling_Std_3,Orders cons. (anon)_Rolling_Mean_6,Orders cons. (anon)_Rolling_Std_6,Orders cons. (anon)_Rolling_Mean_12,Orders cons. (anon)_Rolling_Std_12,Revenue cons. (anon)_Rolling_Mean_3,Revenue cons. (anon)_Rolling_Std_3,Revenue cons. (anon)_Rolling_Mean_6,Revenue cons. (anon)_Rolling_Std_6,Revenue cons. (anon)_Rolling_Mean_12,Revenue cons. (anon)_Rolling_Std_12,China_Core_Inflation_Rate_Lag_1,China_Core_Inflation_Rate_Lag_3,China_Core_Inflation_Rate_MoM_Diff,China_GDP_Lag_1,China_GDP_Lag_3,China_GDP_MoM_Diff,China_GDP_from_Construction_Lag_1,China_GDP_from_Construction_Lag_3,China_GDP_from_Construction_MoM_Diff,China_GDP_from_Manufacturing_Lag_1,China_GDP_from_Manufacturing_Lag_3,China_GDP_from_Manufacturing_MoM_Diff,China_Inflation_Rate_Lag_1,China_Inflation_Rate_Lag_3,China_Inflation_Rate_MoM_Diff,France_Core_Inflation_Rate_Lag_1,France_Core_Inflation_Rate_Lag_3,France_Core_Inflation_Rate_MoM_Diff,France_GDP_Lag_1,France_GDP_Lag_3,France_GDP_MoM_Diff,France_GDP_from_Construction_Lag_1,France_GDP_from_Construction_Lag_3,France_GDP_from_Construction_MoM_Diff,France_GDP_from_Manufacturing_Lag_1,France_GDP_from_Manufacturing_Lag_3,France_GDP_from_Manufacturing_MoM_Diff,France_Inflation_Rate_Lag_1,France_Inflation_Rate_Lag_3,France_Inflation_Rate_MoM_Diff,Germany_Core_Inflation_Rate_Lag_1,Germany_Core_Inflation_Rate_Lag_3,Germany_Core_Inflation_Rate_MoM_Diff,Germany_GDP_Lag_1,Germany_GDP_Lag_3,Germany_GDP_MoM_Diff,Germany_GDP_from_Construction_Lag_1,Germany_GDP_from_Construction_Lag_3,Germany_GDP_